In [1]:
import sys, os, joblib
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, precision_recall_curve
sys.path.append(os.path.abspath('..'))
from scripts.pipeline import classification_engineering


In [2]:
# Extracting the scaled data for the training set
X_train, y_train, scaler, train_spline = classification_engineering('../data/training_data.csv')
print(f'Shutdown threshold distribution: {y_train.value_counts()}')

Shutdown threshold distribution: Shutdown Threshold
0    89125
1     7307
Name: count, dtype: int64


In [3]:
# Loading the best model saved from the script
best_svm = joblib.load('../models/svm_shutdown_classification.joblib')

y_pred = best_svm.predict(X_train)
y_prob = best_svm.decision_function(X_train)

# Evaluation Metrics
print("Classification Report")
print(classification_report(y_train, y_pred, target_names=['Safe Operation' , 'Shutdown Event']))

print("Confusion Matrix")
print(confusion_matrix(y_train, y_pred))

roc_auc = roc_auc_score(y_train, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")

Classification Report
                precision    recall  f1-score   support

Safe Operation       1.00      1.00      1.00     89125
Shutdown Event       0.97      1.00      0.99      7307

      accuracy                           1.00     96432
     macro avg       0.99      1.00      0.99     96432
  weighted avg       1.00      1.00      1.00     96432

Confusion Matrix
[[88917   208]
 [    0  7307]]
ROC-AUC Score: 1.0000


In [4]:
# Out Of fold (OOF) decision scores to prevent data leakage
oof_scores = cross_val_predict(best_svm, X_train, y_train, cv=5, method='decision_function', n_jobs=-1)
# Precision Recall to make sure >= 99% shutdown safety condition
precision, recall, thresholds = precision_recall_curve(y_train, oof_scores)
# Scikit-learn adds an extra element at the end therefore we remove that
precision = precision[:-1]
recall = recall[:-1]

indices = np.where(recall >= 0.99)[0]

if len(indices):
    best_idx = indices[np.argmax(precision[indices])]
    chosen_threshold = thresholds[best_idx]
else:
    chosen_threshold = 0 # Fallback

print(f'SVM Threshold calculated: {chosen_threshold:.4f}')
joblib.dump(chosen_threshold, '../models/svm_threshold.joblib')

SVM Threshold calculated: 0.1976


['../models/svm_threshold.joblib']

In [5]:
# Transforming the 2025 test data usign our scripts
X_test, y_test, _, _ = classification_engineering('../data/testing_data.csv', scaler = scaler, spline = train_spline)
# Generate predictions and decision func
y_test_prob = best_svm.decision_function(X_test)
y_test_pred = (y_test_prob >= chosen_threshold).astype(int)

print("2025 TEST RESULTS:")
print("Classification Report")
print(classification_report(y_test, y_test_pred, target_names=['Safe Operation' , 'Shutdown Event']))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_test_pred))

roc_auc_test = roc_auc_score(y_test, y_test_prob)
print(f"ROC-AUC Score: {roc_auc_test:.4f}")

2025 TEST RESULTS:
Classification Report
                precision    recall  f1-score   support

Safe Operation       1.00      1.00      1.00      7992
Shutdown Event       0.97      1.00      0.98       768

      accuracy                           1.00      8760
     macro avg       0.99      1.00      0.99      8760
  weighted avg       1.00      1.00      1.00      8760

Confusion Matrix
[[7971   21]
 [   3  765]]
ROC-AUC Score: 1.0000
